# Almost Points

This notebook finds every play where a player lost points he reasonably should have had, or gained points he reasonably shouldn't have. This considers, drops, bad throws on open routes, Hail Mary throws, plays stopped just short of the goal line, and teammates that scored touchdowns instead as a result. Nothing is predicted, every value comes from what happened on the field.

Each row is one stat for one player on one play. `points` is what that stat was worth. A positive value is a point the player should have had and didn't get. A negative value is a point he got because of luck. Every row is labeled `luck` (out of his control) or `self_inflicted` (his own mistake).

In [1]:
import re, numpy as np, pandas as pd
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 200)

CURRENT_SZN = 2026

SCORING = {
    "passing_yards": 0.04,
    "passing_tds": 4,
    "interceptions": -2,
    "rushing_yards": 0.1,
    "rushing_tds": 6,
    "receptions": 1,
    "receiving_yards": 0.1,
    "receiving_tds": 6,
    "two_pt_conversions": 2,
    "fumbles_lost": -2,
    "special_teams_tds": 6,
}

pbp = pd.read_parquet(f"data/pbp_{CURRENT_SZN}.parquet")
pbp = pbp[pbp["season_type"] == "REG"].copy()

ftn = pd.read_parquet(f"data/ftn_{CURRENT_SZN}.parquet").rename(
    columns={"nflverse_game_id": "game_id", "nflverse_play_id": "play_id"})
ftn_cols = ["is_drop", "is_catchable_ball", "is_throw_away", "is_contested_ball",
            "is_screen_pass", "is_trick_play", "is_qb_out_of_pocket"]

nullified_ledger = pd.read_parquet(f"data/nullified_ledger_{CURRENT_SZN}.parquet")
nullified_plays = pd.read_parquet(f"data/nullified_plays_{CURRENT_SZN}.parquet")

players = pd.read_parquet(f"data/player_stats_{CURRENT_SZN}.parquet")
positions = players[["player_id", "player_display_name", "position", "position_group"]].drop_duplicates("player_id")

print(f"Plays: {len(pbp):,}  FTN charted plays: {len(ftn):,}")

Plays: 28,467  FTN charted plays: 27,470


## Shared helpers
`event_rows` builds ledger rows for one event. `sign` is +1 when the player lost the stat and -1 when he gained it through luck. Every event uses this same helper, so the whole notebook produces one consistent table.

In [2]:
KEYS = ["season", "week", "game_id", "play_id", "posteam", "series"]

def event_rows(df, player_col, stat, value, event, category, sign=1):
    out = df[KEYS + [player_col]].rename(columns={player_col: "player_id"}).copy()
    out["stat"] = stat
    out["value"] = (df[value].values if isinstance(value, str) else value) * sign
    out["event"] = event
    out["category"] = category
    return out.dropna(subset=["player_id"])

def in_set(df, cols, ids):
    """True for each row whose values in cols appear in ids (a set of tuples)."""
    return np.array([tuple(r) in ids for r in df[cols].itertuples(index=False)], dtype=bool)

# Some names (like "A.St. Brown") aren't matched to an ID by nflfastR. This finds the receiver's
# name after "to 14-" or "intended for 14-" in a description and looks up his ID on that team.
name_ids = players[["player_name", "team", "player_id"]].drop_duplicates(["player_name", "team"])
NAME_IDS = {(n, t): i for n, t, i in name_ids.itertuples(index=False)}
NAME_RE = re.compile(r"(?:to|intended for) \d+-([A-Z][A-Za-z'.\- ]+)")

def receiver_from_text(text, team):
    """Try the longest name first ("A.St. Brown"), then shorter ones, until one matches the team."""
    m = NAME_RE.search(text)
    if not m:
        return np.nan
    words = m.group(1).split(" ")
    for n in range(len(words), 0, -1):
        name = " ".join(words[:n]).rstrip(".")
        if (name, team) in NAME_IDS:
            return NAME_IDS[(name, team)]
    return np.nan

# Counted plays only (nullified plays are handled by notebook 3)
plays = pbp[pbp["play_type"].isin(["pass", "run"]) & (pbp["two_point_attempt"] != 1)].copy()

# Plays changed by replay are handled in the replay cells, so they're kept out of drops and
# open misses (otherwise a catch ruled incomplete could count twice)
reversed_ids = set(zip(*plays.loc[plays["desc"].str.contains("REVERSED.", regex=False, na=False),
                                  ["game_id", "play_id"]].values.T))

# Pass plays joined to FTN charting. Inner join, so plays FTN hasn't charted yet are left out.
passes = plays[(plays["play_type"] == "pass") & plays["receiver_player_id"].notna()].merge(
    ftn[["game_id", "play_id"] + ftn_cols], on=["game_id", "play_id"], how="inner")
passes = passes[~in_set(passes, ["game_id", "play_id"], reversed_ids)]
# Drops and open misses only happen on passes that weren't caught
passes = passes[passes["complete_pass"] != 1]

# The ball reaches the end zone when the air yards cover the distance to the goal line
passes["in_end_zone"] = passes["air_yards"] >= passes["yardline_100"]
passes["target_td"] = passes["in_end_zone"].astype(int)
print(f"Charted targets: {len(passes):,}")

Charted targets: 3,316


## Drops
A drop is worth what the receiver had in his hands: a reception, air yards to that point, and a touchdown if applicable. Yards after the catch are never added as they can't be perfectly predicted and measured. The drop is the receiver's fault, but the QB loses points of no fault of his own.

In [3]:
drops = passes[passes["is_drop"] == True].copy()

drop_rows = pd.concat([
    event_rows(drops, "receiver_player_id", "receptions", 1, "drop", "self_inflicted"),
    event_rows(drops, "receiver_player_id", "receiving_yards", "air_yards", "drop", "self_inflicted"),
    event_rows(drops, "receiver_player_id", "receiving_tds", "target_td", "drop", "self_inflicted"),
    event_rows(drops, "passer_player_id", "passing_yards", "air_yards", "drop", "luck"),
    event_rows(drops, "passer_player_id", "passing_tds", "target_td", "drop", "luck"),
    # drop that became an interception returns the QB's lost points
    event_rows(drops[drops["interception"] == 1], "passer_player_id", "interceptions", 1, "drop", "luck", sign = -1),
], ignore_index = True)
print(f"Drops: {len(drops):,}\nDrops that became interceptions: {(drops["interception"] == 1).sum():,}")

Drops: 417
Drops that became interceptions: 13


## Bad targets
An uncatchable ball only counts when the miss was the QB's fault and the receiver was open. Every uncatchable target gets a label based on FTN charting, checked in this order:
- `throw_away` (removed entirely)
- `drop` a ball that touched his hands was catchable (if no breakup)
- `qb_hit`
- `batted` tagged when a d-lineman got a hand on it at 1 air yard or less
- `defended`
- `contested`
- `hail_mary` derived tag for 40+ air yard throws with 15 seconds or less in the half
- `trick_play` (thrower isn't original QB)
- `open_miss` <- everything else, an open receiver and an uncatchable ball

All count as targets if possible, only `open_miss` affects luck.

In [4]:
uncatch = passes[passes["is_catchable_ball"] == False].merge(
    positions[["player_id", "position_group"]].rename(
        columns = {"player_id": "pass_defense_1_player_id", "position_group": "defender_group"}),
    on = "pass_defense_1_player_id", how = "left")

def bad_target_label(r):
    if r["is_throw_away"] == True:
        return "throw_away"
    if r["is_drop"] == True:
        return "drop"
    if r["qb_hit"] == 1:
        return "qb_hit"
    if pd.notna(r["pass_defense_1_player_id"]):
        if (r["defender_group"] == "DL" and r["air_yards"] <= 1 and r["is_screen_pass"] != True):
            return "batted"
        return "defended"
    if r["is_contested_ball"] == True:
        return "contested"
    if r["air_yards"] >= 40 and r["half_seconds_remaining"] <= 15:
        return "hail_mary"
    if r["is_trick_play"] == True:
        return "trick_play"
    return "open_miss"

uncatch["bad_target_label"] = uncatch.apply(bad_target_label, axis = 1)
print(uncatch["bad_target_label"].value_counts())

bad_target_label
open_miss     885
defended      849
qb_hit        428
contested     217
throw_away    114
batted         89
drop           21
hail_mary       4
trick_play      4
Name: count, dtype: int64


In [5]:
misses = uncatch[uncatch["bad_target_label"] == "open_miss"].copy()

miss_rows = pd.concat([
    event_rows(misses, "receiver_player_id", "receptions", 1, "open_miss", "luck"),
    event_rows(misses, "receiver_player_id", "receiving_yards", "air_yards", "open_miss", "luck"),
    event_rows(misses, "receiver_player_id", "receiving_tds", "target_td", "open_miss", "luck"),
    event_rows(misses, "passer_player_id", "passing_yards", "air_yards", "open_miss", "self_inflicted"),
    event_rows(misses, "passer_player_id", "passing_tds", "target_td", "open_miss", "self_inflicted"),
], ignore_index = True)

# two point tries: drop and misses are worth the 2-point conversion
tp = pbp[(pbp["two_point_attempt"] == 1) & (pbp["play_type"] == "pass") 
    & pbp["receiver_player_id"].notna()].merge(
    ftn[["game_id", "play_id"] + ftn_cols], on = ["game_id", "play_id"], how = "inner")
tp = tp.merge(positions[["player_id", "position_group"]].rename(
    columns = {"player_id": "pass_defense_1_player_id", "position_group": "defender_group"}),
              on = "pass_defense_1_player_id", how = "left")
tp_drops = tp[tp["is_drop"] == True]
tp_unc = tp[(tp["is_catchable_ball"] == False) & (tp["two_point_conv_result"] != "success")].copy()
tp_unc["bad_target_label"] = tp_unc.apply(bad_target_label, axis = 1)
tp_misses = tp_unc[tp_unc["bad_target_label"] == "open_miss"]

two_pt_rows = pd.concat([
    event_rows(tp_drops, "receiver_player_id", "two_pt_conversions", 1, "drop", "self_inflicted"),
    event_rows(tp_drops, "passer_player_id", "two_pt_conversions", 1, "drop", "luck"),
    event_rows(tp_misses, "receiver_player_id", "two_pt_conversions", 1, "open_miss", "luck"),
    event_rows(tp_misses, "passer_player_id", "two_pt_conversions", 1, "open_miss", "self_inflicted"),
], ignore_index = True)
print(f"Two-point drops: {len(tp_drops):,}\nTwo-point open misses: {len(tp_misses):,}")

Two-point drops: 1
Two-point open misses: 12


## Hail Mary
Both outcomes are rare enough to be luck: an interception gives the QB his 2 points back, and a caught TD is 6 lucky points for the receiver (4 for QB), plus yards and the reception.

In [6]:
hm = plays[(plays["play_type"] == "pass") & (plays["air_yards"] >= 40)
           & (plays["half_seconds_remaining"] <= 15)]
hm_int = hm[hm["interception"] == 1]
hm_td = hm[hm["pass_touchdown"] == 1]

hail_rows = pd.concat([
    event_rows(hm_int, "passer_player_id", "interceptions", 1, "hail_mary", "luck", sign=-1),
    event_rows(hm_td, "receiver_player_id", "receiving_tds", 1, "hail_mary", "luck", sign=-1),
    event_rows(hm_td, "receiver_player_id", "receptions", 1, "hail_mary", "luck", sign=-1),
    event_rows(hm_td, "receiver_player_id", "receiving_yards", "receiving_yards", "hail_mary", "luck", sign=-1),
    event_rows(hm_td, "passer_player_id", "passing_tds", 1, "hail_mary", "luck", sign=-1),
    event_rows(hm_td, "passer_player_id", "passing_yards", "passing_yards", "hail_mary", "luck", sign=-1),
], ignore_index=True)
print(f"Hail Mary interceptions: {len(hm_int):,}   Hail Mary TDs: {len(hm_td):,}")

Hail Mary interceptions: 4   Hail Mary TDs: 0


## DPI
Credits receiver with air yards and a catch, credits QB with air yards. TDs when applicable.

In [7]:
dpi = pbp[(pbp["play_type"] == "no_play") & (pbp["penalty_type"] == "Defensive Pass Interference")
          & (pbp["penalty_team"] == pbp["defteam"])].copy()
# Fill in receivers nflfastR couldn't match (like St. Brown) from the description
dpi["receiver_id"] = dpi["receiver_id"].fillna(pd.Series(
    [receiver_from_text(d, t) for d, t in zip(dpi["desc"], dpi["posteam"])], index=dpi.index))

# If the DPI wiped out a catch, notebook 3 already credits the catch he made, so the DPI
# isn't counted on top of it
caught = set(zip(*nullified_plays.loc[nullified_plays["action_type"] == "completion", ["game_id", "play_id"]].values.T))
dpi = dpi[~in_set(dpi, ["game_id", "play_id"], caught)]

dpi_rows = pd.concat([
    event_rows(dpi, "receiver_id", "receptions", 1, "dpi", "luck"),
    event_rows(dpi, "receiver_id", "receiving_yards", "penalty_yards", "dpi", "luck"),
    event_rows(dpi, "passer_id", "passing_yards", "penalty_yards", "dpi", "luck"),
], ignore_index=True)
print(f"DPIs: {len(dpi):,}")

DPIs: 179


## Replay-Overturned TDs

In [8]:
def classify(action):
    if re.search(r"TWO-POINT CONVERSION ATTEMPT", action):
        return "two_point"
    if re.search(r"INTERCEPTED", action):
        return "interception"
    if re.search(r"FUMBLES|Aborted", action):
        return "fumble"
    if re.search(r"sacked", action):
        return "sack"
    if re.search(r"pass incomplete", action):
        return "incomplete"
    if re.search(r" pass ", action):
        return "completion"
    if re.search(r"scrambles", action):
        return "scramble"
    if re.search(r" for (-?\d+ yards?|no gain)", action):
        return "run"
    return "pre_snap"

def parse_yards(action):
    action = re.split(r"INTERCEPTED|FUMBLES", action)[0]
    m = re.search(r" for (-?\d+) yards?| for no gain", action)
    if m is None:
        return np.nan
    return 0 if m.group(1) is None else int(m.group(1))

rev = plays[plays["desc"].str.contains("REVERSED.", regex = False, na = False)].copy()
rev["original"] = rev["desc"].str.split("REVERSED.", n = 1, regex = False).str[0]
rev["corrected"] = rev["desc"].str.split("REVERSED.", n = 1, regex = False).str[1]
rev["original_type"] = rev["original"].apply(classify)

overturned = rev[rev["original"].str.contains("TOUCHDOWN")
    & ~rev["corrected"].str.contains("TOUCHDOWN")
    & rev["original_type"].isin(["completion", "run", "scramble"])].copy()
overturned["original_yards"] = overturned["original"].apply(parse_yards)

#retained stats after reversal
overturned["kept_rec"] = overturned["complete_pass"].fillna(0)
overturned["kept_rec_yds"] = overturned["receiving_yards"].fillna(0)
overturned["kept_rush_yds"] = overturned["rushing_yards"].fillna(0)
overturned["lost_rec"] = 1 - overturned["kept_rec"]
overturned["lost_rec_yds"] = overturned["original_yards"] - overturned["kept_rec_yds"]
overturned["lost_rush_yds"] = overturned["original_yards"] - overturned["kept_rush_yds"]

# The receiver is taken from the original ruling, in case replay changed who it was thrown to
overturned["orig_receiver"] = [receiver_from_text(o, t) for o, t in zip(overturned["original"], overturned["posteam"])]
overturned["orig_receiver"] = overturned["orig_receiver"].fillna(overturned["receiver_player_id"])
ot_pass = overturned[overturned["original_type"] == "completion"]
ot_run = overturned[overturned["original_type"] != "completion"]
ot_run = ot_run.assign(runner = ot_run["rusher_player_id"].fillna(ot_run["passer_player_id"]))

overturn_rows = pd.concat([
    event_rows(ot_pass, "orig_receiver", "receptions", "lost_rec", "overturned_td", "luck"),
    event_rows(ot_pass, "orig_receiver", "receiving_yards", "lost_rec_yds", "overturned_td", "luck"),
    event_rows(ot_pass, "orig_receiver", "receiving_tds", 1, "overturned_td", "luck"),
    event_rows(ot_pass, "passer_player_id", "passing_yards", "lost_rec_yds", "overturned_td", "luck"),
    event_rows(ot_pass, "passer_player_id", "passing_tds", 1, "overturned_td", "luck"),
    event_rows(ot_run, "runner", "rushing_yards", "lost_rush_yds", "overturned_td", "luck"),
    event_rows(ot_run, "runner", "rushing_tds", 1, "overturned_td", "luck"),
], ignore_index=True)
print(f"Overturned TDs: {len(overturned):,}")
overturned[["game_id", "play_id", "desc"]].head()

Overturned TDs: 14


,game_id,play_id,desc
1093,2026_01_DAL_NYG,1372.0,"(5:59) (Shotgun) 33-J.Williams right tackle for 1 yard, TOUCHDOWN. The Replay Official reviewed the runner broke the plane ruling, and the play was REVERSED. (Shotgun) 33-J.Williams right tackle t..."
5381,2026_02_WAS_DAL,1486.0,"(8:01) (Shotgun) 5-J.Daniels pass deep right to 17-T.McLaurin for 34 yards, TOUCHDOWN. The Replay Official reviewed the pass completion ruling, and the play was REVERSED. (Shotgun) 5-J.Daniels pas..."
5984,2026_03_BAL_DAL,3302.0,"(13:22) (Shotgun) 22-D.Henry left guard for 2 yards, TOUCHDOWN. The Replay Official reviewed the runner broke the plane ruling, and the play was REVERSED. (Shotgun) 22-D.Henry left guard to DAL 1 ..."
7082,2026_03_LA_DEN,4657.0,"(:51) (Shotgun) 63-A.Palczewski reported in as eligible. 10-B.Nix right end for 2 yards, TOUCHDOWN. The Replay Official reviewed the runner broke the plane ruling, and the play was REVERSED. (Shot..."
7839,2026_03_PHI_CHI,1003.0,"(14:55) 11-C.Keenum pass short right to 84-C.Loveland for 6 yards, TOUCHDOWN. The Replay Official reviewed the runner broke the plane ruling, and the play was REVERSED. 11-C.Keenum pass short righ..."


## Non-TD replay reversals
a catch ruled incomplete, a spot moved, play ruled a lost fumble, pass ruled an interception, etc

In [9]:
other = rev[~rev["original"].str.contains("TOUCHDOWN")].copy()
other["corrected_type"] = other["corrected"].apply(classify)
other["original_yards"] = other["original"].apply(parse_yards)
other["corrected_yards"] = other["corrected"].apply(parse_yards).fillna(0)
# The receiver is taken from the original ruling, in case replay changed who it was thrown to
other["orig_receiver"] = [receiver_from_text(o, t) for o, t in zip(other["original"], other["posteam"])]
other["orig_receiver"] = other["orig_receiver"].fillna(other["receiver_player_id"])

# Catch ruled incomplete: he loses the reception and every yard
c2i = other[(other["original_type"] == "completion") & (other["corrected_type"] == "incomplete")].copy()
# Same kind of play, fewer yards: he loses the difference
same = other[(other["original_type"] == other["corrected_type"])
             & other["original_type"].isin(["completion", "run", "scramble"])].copy()
same["lost_yards"] = same["original_yards"] - same["corrected_yards"]
same = same[same["lost_yards"] > 0]
same_pass = same[same["original_type"] == "completion"]
same_run = same[same["original_type"] != "completion"]
same_run = same_run.assign(runner=same_run["rusher_player_id"].fillna(same_run["passer_player_id"]))
# A catch and lost fumble ruled incomplete: he loses the catch and its yards, but he also gets out
# of the fumble. If the fumble was lost on the field, those 2 points come off, so the play counts
# at its net value (a short catch can come out negative, meaning replay helped him).
c2i_fum = other[(other["original_type"] == "fumble") & (other["corrected_type"] == "incomplete")
                & other["original"].str.contains(r" pass (?:short|deep) (?:left|middle|right) to ", regex=True)
                & ~other["original"].str.contains("sacked")].copy()
first_recovery = c2i_fum["original"].str.extract(r"FUMBLES.*?RECOVERED by ([A-Z]{2,3})-")[0]
c2i_fum["fumble_was_lost"] = ((first_recovery.notna() & (first_recovery != c2i_fum["posteam"]))
                              | c2i_fum["original"].str.contains("Touchback"))
# Ruled a lost fumble on review
r_fum = other[~other["original"].str.contains("FUMBLES") & other["corrected"].str.contains("FUMBLES")
              & (other["fumble_lost"] == 1)].copy()
# Ruled an interception on review
r_int = other[~other["original"].str.contains("INTERCEPTED") & other["corrected"].str.contains("INTERCEPTED")]

reversal_rows = pd.concat([
    event_rows(c2i, "orig_receiver", "receptions", 1, "replay_reversal", "luck"),
    event_rows(c2i, "orig_receiver", "receiving_yards", "original_yards", "replay_reversal", "luck"),
    event_rows(c2i, "passer_player_id", "passing_yards", "original_yards", "replay_reversal", "luck"),
    event_rows(same_pass, "orig_receiver", "receiving_yards", "lost_yards", "replay_reversal", "luck"),
    event_rows(same_pass, "passer_player_id", "passing_yards", "lost_yards", "replay_reversal", "luck"),
    event_rows(same_run, "runner", "rushing_yards", "lost_yards", "replay_reversal", "luck"),
    event_rows(r_int, "passer_player_id", "interceptions", 1, "replay_reversal", "luck", sign=-1),
    event_rows(c2i_fum, "orig_receiver", "receptions", 1, "replay_reversal", "luck"),
    event_rows(c2i_fum, "orig_receiver", "receiving_yards", "original_yards", "replay_reversal", "luck"),
    event_rows(c2i_fum, "passer_player_id", "passing_yards", "original_yards", "replay_reversal", "luck"),
    event_rows(c2i_fum[c2i_fum["fumble_was_lost"]], "orig_receiver", "fumbles_lost", 1, "replay_reversal", "luck"),
], ignore_index=True)
print(f"Catches ruled incomplete: {len(c2i):,}   Spots moved back: {len(same):,}   "
      f"Ruled lost fumbles: {len(r_fum):,}   Ruled interceptions: {len(r_int):,}   "
      f"Catch-and-fumbles ruled incomplete: {len(c2i_fum):,} ({c2i_fum['fumble_was_lost'].sum()} fumbles lost)")
# The ruled fumbles get their rows in the down-short cell, once the lost-fumble helper exists

Catches ruled incomplete: 22   Spots moved back: 13   Ruled lost fumbles: 12   Ruled interceptions: 2   Catch-and-fumbles ruled incomplete: 4 (3 fumbles lost)


## Penalty-Nullified Plays
From 03

In [10]:
null_rows = nullified_ledger[nullified_ledger["stat"] != "targets"].copy()
null_rows["category"] = np.where(null_rows["penalty_on_self"], "self_inflicted", "luck")
null_rows["event"] = "penalty_nullified"
null_rows = null_rows.merge(nullified_plays[["game_id", "play_id", "posteam", "series"]],
                            on = ["game_id", "play_id"], how = "left")
null_rows = null_rows[KEYS + ["player_id", "stat", "value", "event", "category"]]
print(f"Nullified rows: {len(null_rows):,}")

Nullified rows: 967


## Downfield Penalties
Play still counts up to the spot, so the player loses the yards beyond it (and TD if applicable). 

In [11]:
def on_field_part(desc):
    if "REVERSED." in desc:
        desc = desc.split("REVERSED.", 1)[1]
    return re.split(r"PENALTY|Penalty on", desc, maxsplit=1)[0]

trunc = plays[(plays["penalty"] == 1) & (plays["penalty_team"] == plays["posteam"])
              & ~plays["desc"].str.contains("No Play", na=False)
              & ((plays["complete_pass"] == 1) | (plays["play_type"] == "run"))].copy()
trunc["field_yards"] = trunc["desc"].apply(on_field_part).apply(parse_yards)
trunc["credited"] = trunc["receiving_yards"].fillna(trunc["rushing_yards"])
trunc["lost_yards"] = trunc["field_yards"] - trunc["credited"]
trunc["td_lost"] = trunc["desc"].str.contains("TOUCHDOWN NULLIFIED", na=False)
trunc = trunc[(trunc["lost_yards"] > 0) | trunc["td_lost"]]
trunc = trunc[~in_set(trunc, ["game_id", "play_id"], set(zip(overturned["game_id"], overturned["play_id"])))]
trunc["runner"] = trunc["rusher_player_id"].fillna(trunc["passer_player_id"])

t_pass = trunc[trunc["complete_pass"] == 1]
t_run = trunc[trunc["complete_pass"] != 1]
trunc_rows = pd.concat([
    event_rows(t_pass, "receiver_player_id", "receiving_yards", "lost_yards", "penalty_truncated", "luck"),
    event_rows(t_pass[t_pass["td_lost"]], "receiver_player_id", "receiving_tds", 1, "penalty_truncated", "luck"),
    event_rows(t_pass, "passer_player_id", "passing_yards", "lost_yards", "penalty_truncated", "luck"),
    event_rows(t_pass[t_pass["td_lost"]], "passer_player_id", "passing_tds", 1, "penalty_truncated", "luck"),
    event_rows(t_run, "runner", "rushing_yards", "lost_yards", "penalty_truncated", "luck"),
    event_rows(t_run[t_run["td_lost"]], "runner", "rushing_tds", 1, "penalty_truncated", "luck"),
], ignore_index=True)
# His own penalty is self-inflicted
self_pen = trunc_rows.merge(trunc[["game_id", "play_id", "penalty_player_id"]], on=["game_id", "play_id"])
trunc_rows["category"] = np.where(self_pen["player_id"] == self_pen["penalty_player_id"],
                                  "self_inflicted", "luck")
print(f"Downfield penalty plays: {len(trunc):,}\nTDs lost: {trunc['td_lost'].sum():,}")

Downfield penalty plays: 58
TDs lost: 9


## Down Short
A run or catch that ends at or within the 2-yardline without scoring. The player is credited with a TD and the yards he was short. On a catch, the QB gets the matching passing TD and yards. Plays that started at the 2 or closer are flagged as goal-line stuffs. Plays already counted as an overturned TD or a downfield-penalty TD are left out, so the same TD isn't counted twice.

A lost fumble at the 2 or closer is also down short, but the lsot TD is self-inflicted. The same goes for a fumble a teammate recovers in the end zone, since that TD earns no fantasy points for anyone.

In [12]:
plays["end_yardline"] = plays["yardline_100"] - plays["yards_gained"]
overturned_ids = set(zip(overturned["game_id"], overturned["play_id"]))

short = plays[((plays["complete_pass"] == 1) | (plays["play_type"] == "run"))
              & (plays["touchdown"] != 1) & (plays["fumble_lost"] != 1) & (plays["qb_kneel"] != 1)
              & plays["end_yardline"].between(1, 2)].copy()
# TDs erased by a downfield penalty are counted there, not as down short
trunc_td_ids = set(zip(trunc.loc[trunc["td_lost"], "game_id"], trunc.loc[trunc["td_lost"], "play_id"]))
short = short[~in_set(short, ["game_id", "play_id"], overturned_ids | trunc_td_ids)]
short["is_catch"] = short["complete_pass"] == 1
short["player"] = np.where(short["is_catch"], short["receiver_player_id"],
                           short["rusher_player_id"].fillna(short["passer_player_id"]))
short["goal_line_stuff"] = short["yardline_100"] <= 2
print(f"Down-short plays: {len(short):,}\nGoal-line stuffs: {short['goal_line_stuff'].sum():,}")

# lost fumbles at the 2 or closer. A play can have two fumbles, so the player is whoever's
# fumble was lost (recovered by the other team, or out of the end zone for a touchback).
fum = plays[((plays["complete_pass"] == 1) | (plays["play_type"] == "run"))
            & (plays["touchdown"] != 1) & (plays["fumble_lost"] == 1)
            & plays["end_yardline"].between(1, 2)].copy()
fum = fum[~in_set(fum, ["game_id", "play_id"], overturned_ids)]

def lost_fumbler(r):
    for n in [2, 1]:
        player, team, rec = r[f"fumbled_{n}_player_id"], r[f"fumbled_{n}_team"], r[f"fumble_recovery_{n}_team"]
        if pd.isna(player):
            continue
        if pd.notna(rec) and rec != team:
            return player
        if pd.isna(rec) and "Touchback" in r["desc"]:
            return player
    return np.nan

fum["player"] = fum.apply(lost_fumbler, axis=1)
fum["is_catch"] = fum["player"] == fum["receiver_player_id"]
fum["goal_line_stuff"] = fum["yardline_100"] <= 2

# a fumble near the goal line that a teammate recovers in the end zone. A fumble-recovery TD
# isn't a rushing or receiving TD, so it earns no fantasy points: the fumbler loses the TD and
# nobody gains it. A player who recovers his own fumble for the TD keeps it, so he's skipped.
ftd = plays[(plays["fumble"] == 1) & (plays["touchdown"] == 1) & (plays["td_team"] == plays["posteam"])
            & (plays["rush_touchdown"] != 1) & (plays["pass_touchdown"] != 1)
            & ((plays["complete_pass"] == 1) | (plays["play_type"] == "run"))].copy()
ftd = ftd[ftd["fumbled_1_player_id"] != ftd["td_player_id"]]
ftd["player"] = ftd["fumbled_1_player_id"]
ftd["is_catch"] = ftd["player"] == ftd["receiver_player_id"]
ftd["end_yardline"] = ftd["yardline_100"] - ftd["receiving_yards"].fillna(ftd["rushing_yards"])
ftd = ftd[ftd["end_yardline"].between(0, 2)]
ftd["goal_line_stuff"] = ftd["yardline_100"] <= 2
fum = pd.concat([fum, ftd], ignore_index=True)
print(f"Down-short fumbles: {len(fum):,} (including {len(ftd)} recovered by a teammate for a TD)")

# replay reversals ruled a lost fumble: he gets the 2 points back as luck
r_fum["player"] = r_fum.apply(lost_fumbler, axis=1)
reversal_rows = pd.concat([reversal_rows,
    event_rows(r_fum, "player", "fumbles_lost", 1, "replay_reversal", "luck", sign=-1)], ignore_index=True)

Down-short plays: 331
Goal-line stuffs: 100
Down-short fumbles: 6 (including 1 recovered by a teammate for a TD)


## One lost TD per player per series
A player can lose at most one TD per series, but every player with a real chance to score counts on his own. So a player with 3 stuffed goal-line rushes only loses one potential touchdown as opposed to three, similar cases apply.

Down short plays that aren't counted stay in the play list, but get no points.

In [13]:
s_catch, s_run = short[short["is_catch"]], short[~short["is_catch"]]
short_rows = pd.concat([
    event_rows(s_catch, "player", "receiving_tds", 1, "down_short", "luck"),
    event_rows(s_catch, "player", "receiving_yards", "end_yardline", "down_short", "luck"),
    event_rows(s_catch, "passer_player_id", "passing_tds", 1, "down_short", "luck"),
    event_rows(s_catch, "passer_player_id", "passing_yards", "end_yardline", "down_short", "luck"),
    event_rows(s_run, "player", "rushing_tds", 1, "down_short", "luck"),
    event_rows(s_run, "player", "rushing_yards", "end_yardline", "down_short", "luck"),
], ignore_index = True)

for short fumbles:

* fumbler loses points (his own fault)
* qb loses potential points (not his fault), like a drop

In [14]:
f_catch, f_run = fum[fum["is_catch"]], fum[~fum["is_catch"]]
fumble_rows = pd.concat([
    event_rows(f_catch, "player", "receiving_tds", 1, "down_short_fumble", "self_inflicted"),
    event_rows(f_catch, "player", "receiving_yards", "end_yardline", "down_short_fumble", "self_inflicted"),
    event_rows(f_catch, "passer_player_id", "passing_tds", 1, "down_short_fumble", "luck"),
    event_rows(f_catch, "passer_player_id", "passing_yards", "end_yardline", "down_short_fumble", "luck"),
    event_rows(f_run, "player", "rushing_tds", 1, "down_short_fumble", "self_inflicted"),
    event_rows(f_run, "player", "rushing_yards", "end_yardline", "down_short_fumble", "self_inflicted"),
], ignore_index=True)

In [15]:
# every play where someone had a chance at a TD: a TD taken away, or a down-short play
null_td_ids = set(zip(*nullified_plays.loc[nullified_plays["td_nullified"], ["game_id", "play_id"]].values.T))
trunc_td_ids = set(zip(trunc.loc[trunc["td_lost"], "game_id"], trunc.loc[trunc["td_lost"], "play_id"]))
taken_ids = null_td_ids | overturned_ids | trunc_td_ids

chance_rows = pd.concat([
    null_rows[in_set(null_rows, ["game_id", "play_id"], null_td_ids)],
    overturn_rows,
    trunc_rows[in_set(trunc_rows, ["game_id", "play_id"], trunc_td_ids)],
    short_rows,
    fumble_rows,
], ignore_index = True)
chance_rows["priority"] = np.where(in_set(chance_rows, ["game_id", "play_id"], taken_ids), 1, 0)

# for each player in each series, keep highest priority play, then the latest
SERIES = ["game_id", "posteam", "series"]
picked = (chance_rows[SERIES + ["player_id", "play_id", "priority"]].drop_duplicates()
          .sort_values(["priority", "play_id"])
          .groupby(SERIES + ["player_id"]).tail(1))
picked_ids = set(zip(picked["game_id"], picked["posteam"], picked["series"],
                     picked["player_id"], picked["play_id"]))
counted_rows = chance_rows[in_set(chance_rows, SERIES + ["player_id", "play_id"], picked_ids)]
counted_rows = counted_rows.drop(columns = "priority")

short["counted"] = in_set(short, ["game_id", "play_id"], set(zip(counted_rows["game_id"], counted_rows["play_id"])))
fum["counted"] = in_set(fum, ["game_id", "play_id"], set(zip(counted_rows["game_id"], counted_rows["play_id"])))

# rows from td chance plays are replaced by the counted ones, everything else untouched
null_rows = null_rows[~in_set(null_rows, ["game_id", "play_id"], null_td_ids)]
trunc_rows = trunc_rows[~in_set(trunc_rows, ["game_id", "play_id"], trunc_td_ids)]

chance_series = picked[SERIES].drop_duplicates()
print(f"Series with a lost TD: {len(chance_series):,}\nPlayers charged: {len(picked):,}"
      f"\nDown-short plays counted: {short["counted"].sum():,} of {len(short):,}")

Series with a lost TD: 322
Players charged: 469
Down-short plays counted: 293 of 331


In [16]:
scored = pbp[((pbp["rush_touchdown"] == 1) | (pbp["pass_touchdown"] == 1))
             & (pbp["td_team"] == pbp["posteam"]) & (pbp["two_point_attempt"] != 1)]
scored = scored[["season", "week", "game_id", "posteam", "series", "play_id", "td_player_id",
                 "passer_player_id", "pass_touchdown", "receiving_yards", "rushing_yards", "passing_yards"]]

first_chance = picked.groupby(SERIES, as_index=False)["play_id"].min().rename(columns={"play_id": "first_chance"})
transfers = scored.merge(first_chance, on=SERIES)
transfers = transfers[transfers["play_id"] > transfers["first_chance"]]

pass_t = transfers[transfers["pass_touchdown"] == 1]
run_t = transfers[transfers["pass_touchdown"] != 1]
transfer_rows = pd.concat([
    event_rows(pass_t, "td_player_id", "receiving_tds", 1, "td_transfer", "luck", sign=-1),
    event_rows(pass_t, "td_player_id", "receptions", 1, "td_transfer", "luck", sign=-1),
    event_rows(pass_t, "td_player_id", "receiving_yards", "receiving_yards", "td_transfer", "luck", sign=-1),
    event_rows(pass_t, "passer_player_id", "passing_tds", 1, "td_transfer", "luck", sign=-1),
    event_rows(pass_t, "passer_player_id", "passing_yards", "passing_yards", "td_transfer", "luck", sign=-1),
    event_rows(run_t, "td_player_id", "rushing_tds", 1, "td_transfer", "luck", sign=-1),
    event_rows(run_t, "td_player_id", "rushing_yards", "rushing_yards", "td_transfer", "luck", sign=-1),
], ignore_index=True)
print(f"Transfers: {len(transfers):,}\nTransfer rows: {len(transfer_rows):,}")

Transfers: 139
Transfer rows: 467


## The luck ledger

In [17]:
luck = pd.concat([drop_rows, miss_rows, two_pt_rows, hail_rows, dpi_rows,
                  reversal_rows, null_rows, trunc_rows, counted_rows, transfer_rows],
                 ignore_index=True)

luck["points"] = luck["value"] * luck["stat"].map(SCORING)
unscored = set(luck["stat"]) - set(SCORING)
if unscored:
    print(f"WARNING: no scoring value for {sorted(unscored)}")
if luck["points"].isna().any():
    print(f"WARNING: {luck['points'].isna().sum()} rows have no points")

# Flags for context: thrown on the run (open misses) and goal-line stuffs (down short)
luck = luck.merge(misses[["game_id", "play_id", "is_qb_out_of_pocket"]].rename(
    columns={"is_qb_out_of_pocket": "out_of_pocket"}), on=["game_id", "play_id"], how="left")
luck.loc[luck["event"] != "open_miss", "out_of_pocket"] = np.nan
stuffs = pd.concat([short, fum])[["game_id", "play_id", "goal_line_stuff"]]
luck = luck.merge(stuffs, on=["game_id", "play_id"], how="left")
luck.loc[~luck["event"].isin(["down_short", "down_short_fumble"]), "goal_line_stuff"] = np.nan

luck = luck.merge(positions[["player_id", "player_display_name", "position", "position_group"]],
                  on="player_id", how="left")
print(f"Luck ledger rows: {len(luck):,}")
luck.groupby(["event", "category"])["points"].agg(["count", "sum"]).round(1)


Luck ledger rows: 9,552


count     sum
event             category                     
down_short        luck              778  2186.9
down_short_fumble luck                4     8.1
                  self_inflicted     12    36.8
dpi               luck              534   576.4
drop              luck              848   226.6
                  self_inflicted   1252   839.5
hail_mary         luck                4     8.0
open_miss         luck             2667  2885.3
                  self_inflicted   1782  1006.5
overturned_td     luck               44   119.9
penalty_nullified luck              909   763.2
                  self_inflicted     52    83.0
penalty_truncated luck               73   116.0
                  self_inflicted      4     6.5
replay_reversal   luck              116   108.5
td_transfer       luck              467 -1199.9

In [18]:
# Test
# no player should lose more than one Td, every TD scored after a lost chance should be charged once to the scorer
TD_STATS = ["passing_tds", "receiving_tds", "rushing_tds"]
losses = luck[luck["stat"].isin(TD_STATS) & (luck["value"] > 0)
              & luck["event"].isin(["down_short", "down_short_fumble", "penalty_nullified",
                                    "overturned_td", "penalty_truncated"])]
per_player = losses.groupby(SERIES + ["player_id"])["value"].sum()
print(f"Player-series with a lost TD: {len(per_player):,}\nMore than one: {(per_player > 1).sum():,}")

gains = luck[(luck["event"] == "td_transfer") & luck["stat"].isin(["receiving_tds", "rushing_tds"])]
per_td = gains.groupby(SERIES + ["play_id"])["value"].sum()
print(f"Transfers: {len(per_td):,}\nCharged exactly once: {(per_td == -1).sum():,}")
print(per_player[per_player > 1])

Player-series with a lost TD: 469
More than one: 0
Transfers: 139
Charged exactly once: 139
Series([], Name: value, dtype: float64)


In [19]:
# Test
# a sample from each event is hand checked
import os
if not os.path.exists("data/hand_check_luck.csv"):
    sample_plays = (luck[["event", "game_id", "play_id"]].drop_duplicates()
                    .sample(frac = 1, random_state = CURRENT_SZN)
                    .groupby("event").head(10))
    sample = luck.merge(sample_plays, on = ["event", "game_id", "play_id"])
    sample = sample.merge(pbp[["game_id", "play_id", "desc"]], on = ["game_id", "play_id"], how = "left")
    sample = sample[["event", "category", "game_id", "play_id", "player_display_name", "stat",
    "value", "points", "desc"]].sort_values(["event", "game_id", "play_id"])
    sample["ok"] = ""
    sample.to_csv("data/hand_check_luck.csv", index = False)
    print(f"Saved {sample_plays.shape[0]} plays ({len(sample)} rows) to data/hand_check_luck.csv")
else:
    print(f"data/hand_check_luck.csv already exists")

data/hand_check_luck.csv already exists


In [20]:
summary = luck.pivot_table(index=["player_display_name", "position_group"], columns="category",
                           values="points", aggfunc="sum", fill_value=0)
summary["should_be_total"] = summary.sum(axis=1)
summary.sort_values("luck", ascending=False).head(15).round(1)

,category,luck,self_inflicted,should_be_total
player_display_name,position_group,,,
Denzel Boston,WR,79.3,2.4,81.7
Darnell Mooney,WR,61.1,2.4,63.5
Malik Nabers,WR,56.7,9.6,66.3
Kenneth Walker III,RB,53.5,1.9,55.4
Malik Washington,WR,53.1,0.0,53.1
Jalen Nailor,WR,50.8,4.5,55.3
Michael Wilson,WR,50.4,9.4,59.8
Kyren Williams,RB,50.3,4.5,54.8
Carnell Tate,WR,48.9,6.4,55.3


In [21]:
summary.sort_values("luck").head(10).round(1)

,category,luck,self_inflicted,should_be_total
player_display_name,position_group,,,
Jack Bech,WR,-7.1,1.4,-5.7
Jahdae Walker,WR,-7.1,0.0,-7.1
Mo Alie-Cox,TE,-7.1,1.4,-5.7
Ben Sinnott,TE,-7.1,1.7,-5.4
Dylan Laube,RB,-6.2,0.0,-6.2
Austin Hooper,TE,-6.2,0.0,-6.2
Audric Estime,RB,-6.1,0.0,-6.1
Matthew Hibner,TE,-6.0,2.7,-3.3
Elijah Higgins,TE,-5.9,0.0,-5.9


In [22]:
luck.to_parquet(f"data/luck_ledger_{CURRENT_SZN}.parquet", index=False)
uncatch.to_parquet(f"data/bad_targets_{CURRENT_SZN}.parquet", index=False)
pd.concat([short.assign(fumble=False), fum.assign(fumble=True)]).to_parquet(
    f"data/down_short_{CURRENT_SZN}.parquet", index=False)